# DỰ BÁO ĐỘ MẶN NƯỚC BỀ MẶT BẰNG MÔ HÌNH HỌC MÁY LAI & QUÁ TRÌNH GAUSSIAN KHÔNG GIAN - THỜI GIAN
### Khắc phục triệt để Data Leakage & Tích hợp Spatio-Temporal Residual Gaussian Process
---
**Các điểm cải tiến cốt lõi so với phương pháp cũ:**
1. **Triệt tiêu Data Leakage:** Không sử dụng BMA chọn đặc trưng trên toàn bộ dataset; không chia ngẫu nhiên (Random Row Split) trên chuỗi thời gian.
2. **Phân chia dữ liệu theo Thời gian (Temporal Split):** Sử dụng dữ liệu năm 2019 đến hết tháng 02/2020 làm tập Train (~70%) và từ tháng 03/2020 đến hết tháng 06/2020 làm tập Test độc lập (~30%).
3. **Tái tạo đặc trưng thời gian chu kỳ tuần hoàn:** Giải mã 37 cột nhị phân cũ, loại bỏ 22 dòng khuyết ngày, mã hóa chu kỳ mùa vụ 365 ngày qua cặp biến `sin_doy` và `cos_doy`.
4. **Mô hình lai Không gian - Thời gian:** Dùng các mô hình Cây (Random Forest, Bagging, XGBoost, LightGBM, PSO-RF, PSO-Bagging) học quan hệ phi tuyến viễn thám, kết hợp Quá trình Gaussian (Gaussian Process) học phần sai số (residual) theo thời gian tuyệt đối và theo không gian để bù đắp sai số còn sót lại.
**Bài toán đang giải:** dự báo độ mặn **sau khi đã có ảnh vệ tinh của chính ngày đó**, cho một giai đoạn tương lai chưa từng thấy (train đến hết 2/2020, test từ 3/2020). Không dùng số đo thật của ngày đang dự báo. Hai cách chấm: **tĩnh** (chốt mô hình rồi dự báo cả giai đoạn) và **cuốn chiếu** (được dùng số đo của các cảnh ĐÃ QUA).
6. **Cờ thí nghiệm (Cell 4):** `USE_SEASONAL` (có/không dùng sin/cos_doy), `RECENCY_HALF_LIFE` (trọng số theo độ gần thời gian), `PSO_CV` (fold của PSO), `GP_TIME_LS_MAX` (cận độ dài tương quan thời gian của GP). Nên chạy cả `USE_SEASONAL=True` và `False` rồi báo cáo cả hai.
5. **Đánh giá cuốn chiếu (rolling-origin):** ngoài điểm tĩnh (chốt mô hình rồi dự báo cả tập test một lần), notebook còn chấm theo từng cảnh ảnh: mỗi cảnh test chỉ được hiệu chỉnh bằng số đo thật của các cảnh ĐÃ QUA. Mô hình nền (cây + Ridge) luôn cố định, chỉ GP được cập nhật.

In [ ]:
# Cài đặt các thư viện bổ sung cần thiết: pyswarm, shap, lightgbm, xgboost
!pip install pyswarm shap lightgbm xgboost scikit-learn -q
print('Cài đặt thư viện hoàn tất.')

**Cell 2: Import thư viện & Định nghĩa các hàm đo lường hiệu năng chuẩn của bài báo**
- R2 (Coefficient of Determination - Hệ số xác định)
- RMSE (Root Mean Square Error - Sai số bình phương trung bình căn, đơn vị: g/l hoặc ‰)
- MAE (Mean Absolute Error - Sai số tuyệt đối trung bình, đơn vị: g/l hoặc ‰)
- Ex_VAR (Explained Variance - Phương sai giải thích, đơn vị: %)

In [ ]:
import os
import re
import copy
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.base import clone
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
from pyswarm import pso
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

# 1. Hệ số xác định R2 (Công thức 7 bài báo)
def calc_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

# 2. Sai số bình phương trung bình căn (RMSE - g/l) (Công thức 8 bài báo)
def calc_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# 3. Sai số tuyệt đối trung bình (MAE - g/l) (Công thức 9 bài báo)
def calc_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# 4. Phương sai giải thích (Ex_VAR - %) (Công thức 10 bài báo)
def calc_explained_variance(y_true, y_pred):
    var_y = np.var(y_true)
    if var_y == 0:
        return 0.0
    return (1 - np.var(y_true - y_pred) / var_y) * 100

# 5. Độ lệch trung bình (Bias = mean(y_true - y_pred)); DƯƠNG nghĩa là mô hình đoán THẤP hơn thực tế
def calc_bias(y_true, y_pred):
    return np.mean(y_true - y_pred)

# 6. Hàm tổng hợp các chỉ số. LUÔN gọi theo thứ tự (y_true, y_pred)
def evaluate_model(y_true, y_pred):
    return {
        'RMSE (‰)': round(calc_rmse(y_true, y_pred), 3),
        'MAE (‰)': round(calc_mae(y_true, y_pred), 3),
        'R2': round(calc_r2(y_true, y_pred), 3),
        'Ex_VAR (%)': round(calc_explained_variance(y_true, y_pred), 3),
        'Bias (‰)': round(calc_bias(y_true, y_pred), 3)
    }

print('Import thư viện và định nghĩa các hàm đo lường hoàn tất.')

**Cell 3: Đọc dữ liệu thực tế, giải mã 37 cột ngày và tạo các đặc trưng chu kỳ thời gian tuần hoàn**
- Quét và lọc 37 cột ngày nhị phân cũ (`19Th0119`, `04Th0219`...).
- Phát hiện và loại bỏ 22 dòng khuyết thông tin ngày quan trắc (toàn bộ 37 cột bằng 0).
- Giải mã tên cột ra ngày chuẩn `date`, trích xuất `doy` (Day of Year: 1 - 365), `year`, `month`.
- Tạo 2 biến tuần hoàn `sin_doy` và `cos_doy` cho mô hình cây.
- Xóa sạch 37 cột nhị phân cũ để tối ưu hóa bộ nhớ và triệt tiêu nguy cơ học vẹt.

**3.1 Chia dữ liệu theo thời gian**

In [ ]:

# Danh sách các đường dẫn khả dĩ (Local, Kaggle, Colab)
DATA_PATHS = [
    'Check_BMA_Chuanhoa_LS8_20192020_159.csv',
    './Check_BMA_Chuanhoa_LS8_20192020_159.csv',
    '/kaggle/input/datasets/tienmanhh222/check-bma/Check_BMA_Chuanhoa_LS8_20192020_159.csv',
    '/kaggle/input/check-bma/Check_BMA_Chuanhoa_LS8_20192020_159.csv',
    'Check_BMA_Chuanhoa_LS8_20192020_159.csv'
]

csv_file = None
for p in DATA_PATHS:
    if os.path.exists(p):
        csv_file = p
        break

if csv_file is not None:
    print(f'-> Đang đọc dữ liệu từ: {csv_file}')
    df_raw = pd.read_csv(csv_file)
else:
    raise FileNotFoundError('Không tìm thấy file CSV Check_BMA_Chuanhoa_LS8_20192020_159.csv!')

print(f'-> Kích thước dữ liệu gốc: {df_raw.shape[0]} hàng, {df_raw.shape[1]} cột.')

# 1. Tìm tất cả các cột ngày định dạng ddThmmyy
time_cols = [col for col in df_raw.columns if re.match(r'^\d{2}Th\d{4}$', col)]
print(f'-> Tìm thấy {len(time_cols)} cột ngày nhị phân.')

# 2. Hàm giải mã tên cột (ví dụ: 19Th0119 -> datetime(2019, 1, 19))
def parse_date_column(col_name):
    day = int(col_name[0:2])
    month = int(col_name[4:6])
    year = int('20' + col_name[6:8])
    return datetime(year, month, day)

# 3. Hàm trích xuất ngày an toàn (lọc bỏ các dòng toàn số 0)
def extract_safe_date(row):
    active_cols = [col for col in time_cols if row[col] == 1]
    if len(active_cols) == 1:
        return parse_date_column(active_cols[0])
    return pd.NaT

df_clean = df_raw.copy()
df_clean['date'] = df_clean.apply(extract_safe_date, axis=1)

# 4. Loại bỏ 22 dòng khuyết ngày để dữ liệu sạch 100%
num_missing = df_clean['date'].isna().sum()
df_clean = df_clean.dropna(subset=['date']).reset_index(drop=True)
print(f'-> Đã loại bỏ {num_missing} dòng khuyết ngày. Dữ liệu sạch: {df_clean.shape[0]} mẫu.')

# 5. Tạo các biến thời gian liên tục và tuần hoàn
df_clean['doy'] = df_clean['date'].dt.dayofyear
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['sin_doy'] = np.sin(2 * np.pi * df_clean['doy'] / 365.25)
df_clean['cos_doy'] = np.cos(2 * np.pi * df_clean['doy'] / 365.25)
# 6. Xóa bỏ 37 cột nhị phân cũ
df_clean = df_clean.drop(columns=time_cols)

# 7. Danh sách các đặc trưng viễn thám và địa hình đầy đủ (không lọc qua BMA)
rs_features = [
    'DEM', 'Lat', 'Long', 'EVI', 'NDBI', 'NDMI', 'NDSI', 'NDVI', 'NDWI', 'VSSI', 'SAVI',
    'COSRI', 'EVI2', 'MSI', 'ND23', 'ND47', 'SI1', 'SI2', 'SI3', 'SI4', 'SI5', 'SI6',
    'SI7', 'SI8', 'SI9', 'CA', 'B', 'G', 'R', 'NIR', 'PCA1', 'PCA2', 'PCA3', 'PCA4', 'PCA5'
]

print(f'-> Bảng dữ liệu hoàn tất chuẩn bị: {df_clean.shape[0]} hàng, {df_clean.shape[1]} cột.')
print(df_clean.columns)

**Cell 4: Phân chia tập dữ liệu chống rò rỉ (Temporal Data Splitting - Chuẩn 70% Train / 30% Test)**
- **Tập Huấn luyện (Train Set):** Toàn bộ năm 2019 đến hết ngày 29/02/2020 (~359 mẫu, chiếm ~70.4% dữ liệu).
- **Tập Kiểm thử (Test Set):** Từ ngày 01/03/2020 đến hết tháng 06/2020 (~151 mẫu, chiếm ~29.6% dữ liệu - giai đoạn cao điểm hạn mặn lịch sử).
- **Đặc trưng mô hình Cây (`X_base`):** Toàn bộ 35 biến viễn thám & địa hình + 2 biến tuần hoàn `sin_doy`, `cos_doy` (tổng 37 biến).
- **Đặc trưng Gaussian Process:** Không gian `Lat`, `Long` (đổi sang km, chuẩn hóa bằng thống kê của train) và thời gian tuyệt đối `t` = số ngày kể từ cảnh đầu tiên (KHÔNG dùng `doy`, vì `doy` làm GP coi tháng 3/2019 và tháng 3/2020 là một).
- **Fold chia theo nhóm ngày (`splits`):** tạo một lần ở đây, dùng chung cho PSO, OOF, trọng số blend và phần dư của GP.
- **Nhãn mục tiêu (`y`):** Giữ nguyên nồng độ mặn thực tế `SALINITY_OK` (g/l hoặc ‰).

In [ ]:
target_col = 'SALINITY_OK'

# ===== CỜ THÍ NGHIỆM =====
USE_SEASONAL      = True        # True: dùng sin_doy/cos_doy cho mô hình cây | False: bỏ (nên chạy cả hai và báo cáo cả hai)
RECENCY_HALF_LIFE = None        # None: không đánh trọng số | số ngày (vd 90): hàng càng gần cuối train càng nặng
PSO_CV            = 'forward'   # 'forward': fold tiến theo thời gian (khớp bài toán dự báo) | 'group': GroupKFold theo ngày
GP_TIME_LS_MAX    = 300.0       # cận trên độ dài tương quan thời gian của GP (ngày); thử 60 nếu cảnh cuối mùa bị hiệu chỉnh quá tay
RUN_TAG = f"seasonal{int(USE_SEASONAL)}_hl{RECENCY_HALF_LIFE}_{PSO_CV}"   # gắn vào tên file kết quả

# 1. Phân chia theo mốc thời gian (nguyên cảnh ảnh, không cắt đôi một ngày)
train_mask = (df_clean['date'] >= '2019-01-01') & (df_clean['date'] < '2020-03-01')
test_mask  = (df_clean['date'] >= '2020-03-01')

# Sắp xếp theo ngày (ổn định) RỒI mới reset index, để mọi mảng bên dưới cùng một thứ tự
df_train = df_clean[train_mask].sort_values('date', kind='stable').reset_index(drop=True)
df_test  = df_clean[test_mask].sort_values('date', kind='stable').reset_index(drop=True)

# Thời gian tuyệt đối: số ngày kể từ cảnh đầu tiên (dùng cho GP thay cho doy)
t_0 = df_clean['date'].min()
df_train['t'] = (df_train['date'] - t_0).dt.days
df_test['t']  = (df_test['date']  - t_0).dt.days

# 2. Ma trận đầu vào cho các Mô hình Cây (Viễn thám + Địa hình + sin_doy + cos_doy)
base_cols = rs_features + (['sin_doy', 'cos_doy'] if USE_SEASONAL else [])
X_train_base = df_train[base_cols].values
X_test_base  = df_test[base_cols].values

# 3. Ma trận (Lat, Long, t) – chỉ dùng cho kernel tích tùy chọn ở Cell 6.2
st_cols = ['Lat', 'Long', 't']
X_train_st = df_train[st_cols].values
X_test_st  = df_test[st_cols].values

# 4. Vector biến mục tiêu (Độ mặn thực tế)
y_train = df_train[target_col].values.astype(np.float32)
y_test = df_test[target_col].values.astype(np.float32)

# 5. Fold chia theo NHÓM NGÀY, tạo MỘT LẦN và dùng chung cho PSO / OOF / blend / GP
groups = df_train['date'].dt.strftime('%Y%m%d').values
splits = list(GroupKFold(n_splits=5).split(X_train_base, y_train, groups))
def forward_splits(dates, n_folds=5, min_scenes=10):
    """Fold tiến theo thời gian: mỗi fold chỉ huấn luyện trên các cảnh TRƯỚC khối validation."""
    ud = np.unique(dates) # Lọc ra danh sách các ngày chụp duy nhất không trùng lặp
    edges = np.linspace(min_scenes, len(ud), n_folds + 1).astype(int)
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        tr = np.where(dates < ud[a])[0]
        va = np.where(np.isin(dates, ud[a:b]))[0]
        out.append((tr, va))
    return out

# cross_val_score nhận trực tiếp danh sách (train_idx, val_idx)
cv_strategy = forward_splits(df_train['date'].values) if PSO_CV == 'forward' else splits

# Trọng số theo độ gần thời gian (None -> tất cả bằng 1) và hàm fit có trọng số cho cả Pipeline (Ridge)
def recency_weights(t_days, t_ref, half_life, floor=0.05):
    if half_life is None:
        return np.ones(len(t_days))
    return np.maximum(0.5 ** ((t_ref - np.asarray(t_days, float)) / half_life), floor)

def fit_weighted(model, X, y, w):
    m = clone(model)
    if isinstance(m, Pipeline):
        m.fit(X, y, **{m.steps[-1][0] + '__sample_weight': w})
    else:
        m.fit(X, y, sample_weight=w)
    return m

w_train = recency_weights(df_train['t'].values, df_train['t'].max(), RECENCY_HALF_LIFE)

# 6. In thông số kiểm tra
print('=' * 80)
print('THÔNG TIN PHÂN CHIA TẬP DỮ LIỆU HUẤN LUYỆN & KIỂM THỬ (TEMPORAL SPLIT)')
print('=' * 80)
print(f'-> Tập Train (2019 đến hết T2/2020): {X_train_base.shape[0]} mẫu, {df_train["date"].nunique()} cảnh ({X_train_base.shape[0]/df_clean.shape[0]*100:.1f}%) | Độ mặn TB: {y_train.mean():.2f} ‰')
print(f'-> Tập Test  (T3 đến T6/2020):        {X_test_base.shape[0]} mẫu, {df_test["date"].nunique()} cảnh ({X_test_base.shape[0]/df_clean.shape[0]*100:.1f}%) | Độ mặn TB: {y_test.mean():.2f} ‰')
print(f'-> Số đặc trưng đưa vào mô hình Cây:  {X_train_base.shape[1]} biến (không lọc Lasso)')
print(f'-> Cờ: USE_SEASONAL={USE_SEASONAL} | RECENCY_HALF_LIFE={RECENCY_HALF_LIFE} | PSO_CV={PSO_CV} | GP_TIME_LS_MAX={GP_TIME_LS_MAX}')
print(f'-> Thang đo mặn thực tế: min={y_train.min():.2f}, mean={y_train.mean():.2f}, max={y_train.max():.2f} (‰)')

**Cell 4.1: Kiểm tra nhanh dịch chuyển phân phối và trạm lặp lại (chạy trước khi đọc kết quả mô hình)**
- Mùa test (T3–T6/2020) có mặn hơn hẳn mùa train hay không (so sánh trung bình, p95, tỉ lệ mẫu vượt `max(y_train)`).
- Các trạm test có xuất hiện trong train không (quyết định bạn đang đo "dự báo tương lai tại trạm đã biết" hay "dự báo tại trạm mới").

In [ ]:
print('--- Phân phối độ mặn ---')
print(f'Train: mean={y_train.mean():.2f}  p95={np.percentile(y_train,95):.2f}  max={y_train.max():.2f}')
print(f'Test : mean={y_test.mean():.2f}  p95={np.percentile(y_test,95):.2f}  max={y_test.max():.2f}')
print(f'Tỉ lệ mẫu test vượt max(y_train): {(y_test > y_train.max()).mean():.3f}')

print('\n--- Số mẫu theo cảnh ---')
print('Train:', df_train.groupby('date').size().describe()[['count','min','max']].astype(int).to_dict())
print(df_test.groupby('date').size().to_string())

print('\n--- Trạm lặp lại giữa train và test ---')
_key = lambda d: set(zip(d['Lat'].round(5), d['Long'].round(5)))
tr_pts, te_pts = _key(df_train), _key(df_test)
print(f'Số trạm train: {len(tr_pts)} | test: {len(te_pts)} | trạm test đã có trong train: {len(tr_pts & te_pts)}/{len(te_pts)}')

**Cell 5: Khởi tạo các mô hình cơ sở (Decision Tree, Bagging, Random Forest)**
- 1. Decision Tree Regressor (Cây quyết định mặc định)
- 2. Bagging Regressor cơ sở theo Table 5 bài báo (`n_estimators = 10`)
- 3. Random Forest cơ sở theo Table 5 bài báo (`n_estimators = 100`)
- (1D-CNN được chuyển xuống Cell 7.1 và tắt mặc định, vì tích chập trên trục đặc trưng của dữ liệu bảng không có ý nghĩa.)

In [ ]:
# 1. Decision Tree cơ sở
dt_base = DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42)

# 2. Bagging Regressor cơ sở theo bài báo (Table 5)
bagging_base = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42),
    n_estimators=10,
    random_state=42,
    n_jobs=-1
)

# 3. Random Forest cơ sở theo bài báo (Table 5)
rf_base = RandomForestRegressor(
    n_estimators=100,
    max_depth=7,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

print('Đã khởi tạo xong các mô hình cơ sở: Decision Tree, Bagging, Random Forest.')

**Cell 6: Tối ưu hóa siêu tham số bằng PSO (PSO-RF, PSO-Bagging, PSO-XGBoost, PSO-LightGBM)**
- Đặt cờ `USE_PRESET_OPTIMAL_PARAMS = True` để nạp ngay cấu hình tối ưu chuẩn bài báo (chạy nhanh trong 0.1 giây).
- Đặt `USE_PRESET_OPTIMAL_PARAMS = False` nếu muốn chạy lại thuật toán PSO dò tìm siêu tham số trên tập Train qua 5-Fold Cross Validation.

In [ ]:
# cv_strategy được tạo ở Cell 4 theo cờ PSO_CV (fold tiến hoặc GroupKFold theo ngày)

#Toi uu cho RF
def pso_obj_rf(params):
    n_estimators = int(round(params[0]))
    depth = int(round(params[1]))
    leaf = int(round(params[2]))
    feat = float(params[3])
    min_sample_split = int(round(params[4]))

    model = RandomForestRegressor(
        n_estimators = n_estimators,
        max_depth= depth,
        min_samples_leaf= leaf,
        max_features= feat,
        random_state= 42,
        min_samples_split= min_sample_split
    )
    scores = cross_val_score(model, X_train_base, y_train, cv = cv_strategy, scoring= 'neg_mean_squared_error', n_jobs= -1)
    return -scores.mean()

lb_rf = [120, 3, 3, 0.5, 8]
ub_rf = [250, 8, 10, 0.8, 20]

#Toi uu cho Bagging 
def pso_obj_bag(params):
    n_est = int(round(params[0]))
    depth = int(round(params[1]))
    leaf = int(round(params[2]))
    feat = float(params[3])
    min_sample_split = int(round(params[4]))
    tree = DecisionTreeRegressor(max_depth= depth, min_samples_leaf=leaf, min_samples_split= min_sample_split
    , random_state= 28)

    model = BaggingRegressor(estimator= tree, n_estimators= n_est, max_features= feat, bootstrap= True,
    random_state= 28, n_jobs= -1)
    scores = cross_val_score(model, X_train_base, y_train, cv = cv_strategy,
    scoring= 'neg_mean_squared_error', n_jobs= -1)
    return -scores.mean()

lb_bag = [120, 3, 3, 0.5, 8]
ub_bag = [250, 8, 10, 0.8, 20]

#Toi uu cho XGB
# XGBoost: [n_est, max_depth, lr, subsample, colsample, min_child_weight, gamma, reg_lambda, reg_alpha]
lb_xgb = [50, 2, 0.01, 0.6, 0.5, 3, 0.0, 1.0, 0.0]
ub_xgb = [200, 5, 0.15, 0.85, 0.8, 15, 5.0, 15.0, 5.0]
def pso_obj_xgb(p):
    model = XGBRegressor(
        n_estimators=int(round(p[0])), max_depth=int(round(p[1])),
        learning_rate=p[2], subsample=p[3], colsample_bytree=p[4],
        min_child_weight=p[5], gamma=p[6], reg_lambda=p[7], reg_alpha=p[8],
        random_state=42, n_jobs=-1)
    s = cross_val_score(model, X_train_base, y_train, cv=cv_strategy,
                        scoring='neg_mean_squared_error', n_jobs=1)
    return -s.mean()
# LightGBM: [n_est, num_leaves, lr, subsample, colsample, min_child_samples, reg_lambda, reg_alpha]
lb_lgbm = [50, 5, 0.01, 0.6, 0.5, 10,  1.0, 0.0]
ub_lgbm = [200, 15, 0.15, 0.85, 0.8, 30, 15.0, 5.0]

def pso_obj_lgbm(p):
    model = LGBMRegressor(
        n_estimators=int(round(p[0])), num_leaves=int(round(p[1])), max_depth=-1,
        learning_rate=p[2], subsample=p[3], subsample_freq=1, colsample_bytree=p[4],
        min_child_samples=int(round(p[5])), reg_lambda=p[6], reg_alpha=p[7],
        random_state=42, verbose=-1, n_jobs=-1)
    s = cross_val_score(model, X_train_base, y_train, cv=cv_strategy,
                        scoring='neg_mean_squared_error', n_jobs=1)
    return -s.mean()

lb_rigde = [0.1]
ub_rigde = [200.0]
def pso_obj_rigde(p):
    model = make_pipeline(StandardScaler(), Ridge(alpha= float(p[0]), random_state= 42))
    s = cross_val_score(model, X_train_base, y_train, cv = cv_strategy
    , scoring= 'neg_mean_squared_error', n_jobs= -1)

    return -s.mean()

In [ ]:
# CỜ ĐIỀU KHIỂN: Sử dụng ngay bộ siêu tham số tối ưu chuẩn xác
USE_PRESET_OPTIMAL_PARAMS = False

def extract_pso_result(res):
    """Trích xuất (xopt, fopt) tương thích mọi phiên bản pyswarm/scipy OptimizeResult"""
    if hasattr(res, 'x') and hasattr(res, 'fun'):
        return np.asarray(res.x, dtype=float), float(res.fun)
    elif isinstance(res, dict) and 'x' in res and 'fun' in res:
        return np.asarray(res['x'], dtype=float), float(res['fun'])
    elif isinstance(res, (tuple, list)):
        return np.asarray(res[0], dtype=float), float(res[1])
    return res, None

if USE_PRESET_OPTIMAL_PARAMS:
    print('=== SỬ DỤNG BỘ SIÊU THAM SỐ TỐI ƯU CÔNG BỐ TRONG TABLE 5 BÀI BÁO ===')
    # 1. Cấu hình PSO_Random Forest (RF_PSO trong Table 5 bài báo):
    pso_rf_model = RandomForestRegressor(
        n_estimators=250,
        max_depth=17,
        min_samples_split=2,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    
    # 2. Cấu hình PSO_Bagging (Bagging_PSO trong Table 5 bài báo):
    pso_bag_tree = DecisionTreeRegressor(
        max_depth=19,
        min_samples_split=2,
        min_samples_leaf=5,
        random_state=42
    )
    pso_bagging_model = BaggingRegressor(
        estimator=pso_bag_tree,
        n_estimators=259,
        random_state=42,
        n_jobs=-1
    )
    
    # 3. Cấu hình PSO_XGBoost tối ưu
    pso_xgb_model = XGBRegressor(
        n_estimators=180,
        max_depth=6,
        learning_rate=0.03,
        min_child_weight=4,
        colsample_bytree=0.7,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )
    
    # 4. Cấu hình PSO_LightGBM tối ưu
    pso_lgbm_model = LGBMRegressor(
        n_estimators=160,
        max_depth=7,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=10,
        colsample_bytree=0.7,
        subsample=0.8,
        subsample_freq=1,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )

    # 5. Ridge (preset): alpha mặc định, chỉ để notebook chạy được khi bật cờ preset
    pso_linear_rigde = make_pipeline(StandardScaler(), Ridge(alpha=10.0, random_state=42))
else:
    print('Bắt đầu chạy thuật toán PSO tìm siêu tham số trên tập Train...')
    res_rf = pso(pso_obj_rf, lb_rf, ub_rf, swarmsize=12, maxiter=10)
    xopt_rf, fopt_rf = extract_pso_result(res_rf)
    print(f"RF best MSE: {fopt_rf:.4f}")
    print(f"n_estimators : {int(round(xopt_rf[0]))}")
    print(f"max_depth: {int(round(xopt_rf[1]))}")
    print(f"min_samples_leaf: {int(round(xopt_rf[2]))}")
    print(f"max_features: {float(xopt_rf[3]):.3f}")
    print(f"min_samples_split: {int(round(xopt_rf[4]))}")

    pso_rf_model = RandomForestRegressor(
        n_estimators=int(round(xopt_rf[0])),
        max_depth=int(round(xopt_rf[1])),
        min_samples_leaf=int(round(xopt_rf[2])),
        max_features=float(xopt_rf[3]),
        min_samples_split=int(round(xopt_rf[4])),
        random_state=42,
        n_jobs=-1
    )

    res_bag = pso(pso_obj_bag, lb_bag, ub_bag, swarmsize=12, maxiter=10)
    xopt_bag, fopt_bag = extract_pso_result(res_bag)
    print(f"Bagging best MSE: {fopt_bag:.4f}")
    print(f"n_estimators : {int(round(xopt_bag[0]))}")
    print(f"max_depth: {int(round(xopt_bag[1]))}")
    print(f"min_sample_leaf : {int(round(xopt_bag[2]))}")
    print(f"max_features: {float(xopt_bag[3])}")
    print(f"min_sample_split: {int(round(xopt_bag[4]))}")

    pso_bag_tree = DecisionTreeRegressor(
        max_depth=int(round(xopt_bag[1])),
        min_samples_leaf=int(round(xopt_bag[2])),
        min_samples_split=int(round(xopt_bag[4])),
        random_state=42
    )
    pso_bagging_model = BaggingRegressor(
        estimator=pso_bag_tree,
        n_estimators=int(round(xopt_bag[0])),
        max_features=float(xopt_bag[3]),
        random_state=42,
        n_jobs=-1
    )

    res_xgb = pso(pso_obj_xgb, lb_xgb, ub_xgb, swarmsize=12, maxiter=10)
    xopt_xgb, fopt_xgb = extract_pso_result(res_xgb)
    print(f"XGB best MSE: {fopt_xgb:.4f}")
    print(f"n_estimators: {int(round(xopt_xgb[0]))}")
    print(f"max_depth: {int(round(xopt_xgb[1]))}")
    print(f"learning rate : {float(xopt_xgb[2])}")
    print(f"min_child_weight : {float(xopt_xgb[5])}")
    pso_xgb_model = XGBRegressor(
        n_estimators=int(round(xopt_xgb[0])),
        max_depth=int(round(xopt_xgb[1])),
        learning_rate=float(xopt_xgb[2]),
        subsample=float(xopt_xgb[3]),
        colsample_bytree=float(xopt_xgb[4]),
        min_child_weight=float(xopt_xgb[5]),
        gamma=float(xopt_xgb[6]),
        reg_lambda=float(xopt_xgb[7]),
        reg_alpha=float(xopt_xgb[8]),
        random_state=42,
        n_jobs=-1
    )

    res_lgbm = pso(pso_obj_lgbm, lb_lgbm, ub_lgbm, swarmsize=12, maxiter=10)
    xopt_lgbm, fopt_lgbm = extract_pso_result(res_lgbm)
    print(f"LGBM best MSE: {fopt_lgbm:.4f}")
    print(f"n_estimators : {int(round(xopt_lgbm[0]))}")
    print(f"num_leaves: {int(round(xopt_lgbm[1]))}")
    print(f"learning rate : {float(xopt_lgbm[2])}")
    print(f"min_child_samples : {int(round(xopt_lgbm[5]))}")
    pso_lgbm_model = LGBMRegressor(
        n_estimators=int(round(xopt_lgbm[0])),
        num_leaves=int(round(xopt_lgbm[1])),
        max_depth=-1,
        learning_rate=float(xopt_lgbm[2]),
        subsample=float(xopt_lgbm[3]),
        subsample_freq=1,
        colsample_bytree=float(xopt_lgbm[4]),
        min_child_samples=int(round(xopt_lgbm[5])),
        reg_lambda=float(xopt_lgbm[6]),
        reg_alpha=float(xopt_lgbm[7]),
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )

    res_rigde = pso(pso_obj_rigde, lb_rigde, ub_rigde, swarmsize= 12, maxiter= 10)
    xopt_rigde, fopt_rigde = extract_pso_result(res_rigde)
    print(f"Rigde best MSE: {fopt_rigde:.4f}")
    print(f"Alpha toi uu: {xopt_rigde[0]:.4f}")
    
    pso_linear_rigde = make_pipeline(StandardScaler(), Ridge(alpha= float(xopt_rigde[0]), random_state= 42))
print('Đã cấu hình xong các mô hình: PSO-RF, PSO-Bagging, PSO-XGBoost, PSO-LightGBM và Ridge.')

**Cell 6.0: Chọn `RECENCY_HALF_LIFE` bằng fold tiến (chỉ dùng train)**
Với mỗi fold tiến, trọng số được tính theo khoảng cách tới đầu khối validation, nên mọi fold đều đo được tác dụng của việc ưu tiên dữ liệu gần. Điểm test chỉ được in MỘT lần, cho cấu hình đã được CV chọn (và cho `None` để đối chiếu), không dùng để chọn.
Lưu ý: fold tiến trong train KHÔNG thể đánh giá `sin/cos_doy` một cách công bằng (train chỉ có một mùa lặp lại), nên cờ `USE_SEASONAL` không chọn bằng cell này mà báo cáo cả hai.

In [ ]:
tuned = {'PSO RF': pso_rf_model, 'PSO Bagging': pso_bagging_model,
         'PSO XGB': pso_xgb_model, 'PSO LightGBM': pso_lgbm_model}
t_days = df_train['t'].values
fs = forward_splits(df_train['date'].values)
grid = [None, 180, 90, 45]
order = ['None', '180', '90', '45']

rows = []
for name, mdl in tuned.items():
    for hl in grid:
        sc, bs = [], []
        for tr, va in fs:
            w = recency_weights(t_days[tr], t_days[va].min(), hl)
            m = fit_weighted(mdl, X_train_base[tr], y_train[tr], w)
            p = np.clip(m.predict(X_train_base[va]), 0, None)
            sc.append(calc_r2(y_train[va], p)); bs.append(calc_bias(y_train[va], p))
        rows.append({'Model': name, 'half_life': 'None' if hl is None else str(hl),
                     'CV_R2': np.mean(sc), 'CV_Bias': np.mean(bs)})
cv_hl = pd.DataFrame(rows)
print('R2 trung bình qua các fold tiến (train):')
display(cv_hl.pivot(index='Model', columns='half_life', values='CV_R2')[order].round(3))
print('Bias trung bình qua các fold tiến (train):')
display(cv_hl.pivot(index='Model', columns='half_life', values='CV_Bias')[order].round(3))

mean_by_hl = cv_hl.groupby('half_life')['CV_R2'].mean().reindex(order)
print('R2 trung bình qua 4 mô hình:', mean_by_hl.round(3).to_dict())
chosen = mean_by_hl.idxmax()
chosen_hl = None if chosen == 'None' else float(chosen)

print(f'\nCV chọn half_life = {chosen}. Kiểm tra trên test (một lần, chỉ để tham khảo):')
for hl in ([None] if chosen_hl is None else [None, chosen_hl]):
    w = recency_weights(t_days, t_days.max(), hl)
    out = {}
    for name, mdl in tuned.items():
        m = fit_weighted(mdl, X_train_base, y_train, w)
        ev = evaluate_model(y_test, np.clip(m.predict(X_test_base), 0, None))
        out[name] = f"R2={ev['R2']}, ExVar={ev['Ex_VAR (%)']}, Bias={ev['Bias (‰)']}"
    print(f'  half_life={hl}:')
    for k, v in out.items():
        print(f'     {k:13s} {v}')
print(f"\n-> Muốn dùng: đặt RECENCY_HALF_LIFE = {chosen_hl} ở Cell 4, hoặc chỉ cần chạy lại dòng "
      f"w_train = recency_weights(df_train['t'].values, df_train['t'].max(), {chosen_hl}) rồi chạy từ Cell 7 (không cần chạy lại PSO).")

**Cell 6.1: Gaussian Process học phần sai số (residual) theo thời gian và không gian**

Phần dư của mô hình nền gồm hai thành phần: (a) mức lệch chung của cả một giai đoạn (ví dụ cả mùa 2020 đều bị đoán thấp), (b) khác biệt ổn định giữa các trạm. Một kernel tích `k_s * k_t` chỉ cho hai điểm tương quan khi vừa gần nhau về không gian vừa gần nhau về thời gian, nên khó nắm thành phần (a). Vì vậy GP được tách thành hai phần, dùng kernel có sẵn của sklearn (gradient do sklearn tự tính):
- **GP thời gian:** đầu vào là `t` (số ngày tuyệt đối), kernel `C * Matern(nu=1.5) + White`. Độ dài tương quan đọc thẳng bằng ngày, không tuần hoàn nên phân biệt được 2019 và 2020.
- **GP không gian:** đầu vào là (x, y) theo km đã chuẩn hóa, kernel `C * RBF(ARD) + White`, học phần còn lại sau khi trừ phần thời gian.
- **Dự báo hiệu chỉnh:** `y_hat_final = y_hat_base + e_hat_gp`.

Kernel tích theo công thức của thầy (đã sửa lỗi) vẫn được giữ ở **Cell 6.2** như một lựa chọn, không dùng trong pipeline mặc định.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C, WhiteKernel

# ---------- Đầu vào GP: thời gian tuyệt đối (ngày) + tọa độ km đã chuẩn hóa ----------
lat0, lon0 = df_train['Lat'].mean(), df_train['Long'].mean()      # chỉ lấy từ train

def get_t(d):
    return d['t'].values.astype(float)[:, None]

def get_xy(d):
    x = (d['Long'].values - lon0) * 111.32 * np.cos(np.radians(lat0))   # km
    y = (d['Lat'].values  - lat0) * 110.57                                # km
    return np.c_[x, y]

t_tr,  t_te  = get_t(df_train),  get_t(df_test)
xy_tr, xy_te = get_xy(df_train), get_xy(df_test)
xy_sd = xy_tr.std(axis=0)                       # thống kê chuẩn hóa chỉ từ train
print(f'Độ lệch chuẩn tọa độ (km): x={xy_sd[0]:.1f}, y={xy_sd[1]:.1f}')
xy_tr, xy_te = xy_tr / xy_sd, xy_te / xy_sd

def fit_correction(t, xy, r, n_restarts=3, seed=42):
    """Học phần dư r theo thời gian t (GP 1) rồi theo không gian xy (GP 2)."""
    gp_t = GaussianProcessRegressor(
        C(1.0, (1e-2, 1e3)) * Matern(length_scale=min(45.0, GP_TIME_LS_MAX), length_scale_bounds=(15.0, GP_TIME_LS_MAX), nu=1.5)
        + WhiteKernel(1.0, (1e-2, 1e3)),
        n_restarts_optimizer=n_restarts, random_state=seed).fit(t, r)
    r2 = r - gp_t.predict(t)
    gp_s = GaussianProcessRegressor(
        C(1.0, (1e-2, 1e3)) * RBF([1.0, 1.0], (0.1, 20.0))
        + WhiteKernel(1.0, (1e-2, 1e3)),
        n_restarts_optimizer=n_restarts, random_state=seed).fit(xy, r2)
    return gp_t, gp_s

def predict_correction(gps, t_new, xy_new):
    gp_t, gp_s = gps
    return gp_t.predict(t_new) + gp_s.predict(xy_new)

print('Đã định nghĩa GP hai phần (thời gian + không gian).')

**Cell 6.2 (tùy chọn): Kernel tích không - thời gian theo công thức của thầy, đã sửa lỗi**
Không được gọi trong pipeline mặc định. Nếu muốn dùng, thay `fit_correction` bằng một `GaussianProcessRegressor(kernel=st_kernel + WhiteKernel(...))` fit trên cột `(Lat, Long, t)`. Lưu ý kernel này tuần hoàn thuần (chu kỳ 365.25 ngày) nên hai ngày cách nhau 366 ngày vẫn tương quan gần 1; muốn phân biệt 2019 với 2020 phải nhân thêm một thành phần không tuần hoàn theo thời gian.

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.gaussian_process.kernels import Kernel, Hyperparameter

class SpatioTemporalProductKernel(Kernel):
    def __init__(self, l_s=0.2, l_t=1.0, sigma_f=1.0,
                 l_s_bounds=(0.01, 5.0), l_t_bounds=(0.1, 5.0), sigma_f_bounds=(0.1, 50.0)):
        self.l_s = l_s
        self.l_t = l_t
        self.sigma_f = sigma_f
        self.l_s_bounds = l_s_bounds
        self.l_t_bounds = l_t_bounds
        self.sigma_f_bounds = sigma_f_bounds

    @property
    def hyperparameter_l_s(self):
        return Hyperparameter('l_s', 'numeric', self.l_s_bounds)

    @property
    def hyperparameter_l_t(self):
        return Hyperparameter('l_t', 'numeric', self.l_t_bounds)

    @property
    def hyperparameter_sigma_f(self):
        return Hyperparameter('sigma_f', 'numeric', self.sigma_f_bounds)

    def __call__(self, X, Y=None, eval_gradient=False):
        X = np.atleast_2d(X)
        Y_ = X if Y is None else np.atleast_2d(Y)
        X_s, Y_s = X[:, :2], Y_[:, :2]
        X_t, Y_t = X[:, 2:3], Y_[:, 2:3]

        d_s_sq = cdist(X_s, Y_s, metric='sqeuclidean')
        k_s = np.exp(-d_s_sq / (2 * self.l_s ** 2))

        delta_t = cdist(X_t, Y_t, metric='cityblock')
        sin_term = np.sin(np.pi * delta_t / 365.25) ** 2
        k_t = np.exp(-2 * sin_term / self.l_t ** 2)

        K = self.sigma_f ** 2 * (k_s * k_t)

        if eval_gradient:
            # gradient theo log-siêu-tham-số (sklearn tối ưu trong không gian log)
            grad_l_s = K * d_s_sq / self.l_s ** 2
            grad_l_t = K * 4 * sin_term / self.l_t ** 2
            grad_sigma_f = 2.0 * K
            return K, np.dstack((grad_l_s, grad_l_t, grad_sigma_f))
        return K

    def diag(self, X):
        return np.full(X.shape[0], self.sigma_f ** 2)

    def is_stationary(self):
        return True

st_kernel = SpatioTemporalProductKernel(l_s=0.2, l_t=1.0, sigma_f=1.0)

**Cell 7: Huấn luyện các mô hình Cây + Ridge, blend, và GP học phần dư OOF**
- Mọi OOF (cây, Ridge, CNN nếu bật) dùng CHUNG danh sách `splits` (fold theo nhóm ngày).
- Trọng số blend cây/Ridge tìm bằng stacking trên OOF (`BLEND_METHOD`).
- GP học phần dư OOF của chính dự báo đã blend (`y_train - oof_blend`), không dùng phần dư in-sample.
- Bảng kết quả có thêm `CV_R2 (OOF)`: đây mới là con số đáng so sánh với Test_R2. `Train_R2` là in-sample nên không có giá trị so sánh.
- Có hai dòng baseline (trung bình train, Ridge đơn lẻ) để biết R² hiện tại tốt hay xấu so với mốc đơn giản.

In [ ]:
# splits (GroupKFold theo nhóm ngày) đã tạo ở Cell 4 – dùng chung cho mọi thứ
def oof_predict(model):
    oof = np.zeros(len(y_train), dtype=float)
    for tr, va in splits:
        m = fit_weighted(model, X_train_base[tr], y_train[tr], w_train[tr])
        oof[va] = m.predict(X_train_base[va])
    return oof

def get_blend_weights(oof_tree, oof_lin, y_true, method='stacking'):
    """Trọng số (w_tree, w_lin), tổng bằng 1, chỉ dựa trên dự báo OUT-OF-FOLD."""
    from scipy.optimize import minimize_scalar
    oof_tree = np.asarray(oof_tree, float); oof_lin = np.asarray(oof_lin, float)
    y_true = np.asarray(y_true, float)
    if method == 'stacking':
        f = lambda w: np.mean((y_true - (w * oof_tree + (1 - w) * oof_lin)) ** 2)
        w_tree = float(minimize_scalar(f, bounds=(0.0, 1.0), method='bounded').x)
    elif method == 'inv_mse':
        inv = np.array([1 / mean_squared_error(y_true, oof_tree), 1 / mean_squared_error(y_true, oof_lin)])
        w_tree = float((inv / inv.sum())[0])
    else:
        raise ValueError("method phải là 'stacking' hoặc 'inv_mse'")
    return w_tree, 1.0 - w_tree

BLEND_METHOD = 'stacking'     # 'stacking' (khuyến nghị) hoặc 'inv_mse'

models_tree = {
    'Decision Tree': dt_base, 'Bagging': bagging_base, 'PSO Bagging': pso_bagging_model,
    'Random Forest': rf_base, 'PSO RF': pso_rf_model,
    'PSO XGB': pso_xgb_model, 'PSO LightGBM': pso_lgbm_model,
}

def row(name, ev_tr, ev_cv, ev_te):
    return {
        'Model': name,
        'Train_R2 (in-sample)': ev_tr['R2'],
        'CV_R2 (OOF)': ev_cv['R2'], 'CV_RMSE (‰)': ev_cv['RMSE (‰)'],
        'Test_RMSE (‰)': ev_te['RMSE (‰)'], 'Test_MAE (‰)': ev_te['MAE (‰)'],
        'Test_R2': ev_te['R2'], 'Test_Ex_VAR (%)': ev_te['Ex_VAR (%)'],
        'Test_Bias (‰)': ev_te['Bias (‰)'],
    }

# ---- Ridge: OOF + fit full + dự báo, làm 1 lần ----
oof_linear = oof_predict(pso_linear_rigde)
ridge_full = fit_weighted(pso_linear_rigde, X_train_base, y_train, w_train)
pred_linear_train = ridge_full.predict(X_train_base)
pred_linear_test  = ridge_full.predict(X_test_base)

results, predictions, store = [], {}, {}

# ---- Baseline đơn giản để biết R2 hiện tại tốt hay xấu ----
oof_mean = np.zeros(len(y_train))
for tr, va in splits:
    oof_mean[va] = y_train[tr].mean()
results.append(row('Baseline: trung bình train',
                   evaluate_model(y_train, np.full(len(y_train), y_train.mean())),
                   evaluate_model(y_train, oof_mean),
                   evaluate_model(y_test,  np.full(len(y_test),  y_train.mean()))))
results.append(row('Baseline: Ridge đơn lẻ',
                   evaluate_model(y_train, np.clip(pred_linear_train, 0, None)),
                   evaluate_model(y_train, np.clip(oof_linear, 0, None)),
                   evaluate_model(y_test,  np.clip(pred_linear_test, 0, None))))

# ---- Vòng lặp các mô hình cây ----
for name, model in models_tree.items():
    print(f"Đang xử lý: {name}")

    # 1) OOF của cây + trọng số blend (cùng fold với Ridge)
    oof_tree = oof_predict(model)
    w_tree, w_lin = get_blend_weights(oof_tree, oof_linear, y_train, method=BLEND_METHOD)
    print(f"  Trọng số: Tree = {w_tree:.2f}, Ridge = {w_lin:.2f}")
    oof_blend = np.clip(w_tree * oof_tree + w_lin * oof_linear, 0, None)    # độ mặn không âm

    # 2) Fit full train, dự báo (nhớ nhân w_tree cho nhánh cây)
    tree_full = fit_weighted(model, X_train_base, y_train, w_train)
    pred_train_base = np.clip(w_tree * tree_full.predict(X_train_base) + w_lin * pred_linear_train, 0, None)
    pred_test_base  = np.clip(w_tree * tree_full.predict(X_test_base)  + w_lin * pred_linear_test,  0, None)

    ev_tr = evaluate_model(y_train, pred_train_base)
    ev_cv = evaluate_model(y_train, oof_blend)
    ev_te = evaluate_model(y_test,  pred_test_base)
    results.append(row(name, ev_tr, ev_cv, ev_te))

    # 3) GP học phần dư OOF CỦA CHÍNH DỰ BÁO BLEND (đánh giá tĩnh: GP chỉ thấy train)
    residual_oof = y_train - oof_blend
    gps = fit_correction(t_tr, xy_tr, residual_oof)
    print(f"  GP thời gian: {gps[0].kernel_}")
    print(f"  GP không gian: {gps[1].kernel_}")
    pred_test_hybrid = np.clip(pred_test_base + predict_correction(gps, t_te, xy_te), 0, None)
    results.append(row(f"{name} + GP", ev_tr, ev_cv, evaluate_model(y_test, pred_test_hybrid)))

    predictions[name] = {'base': pred_test_base, 'hybrid': pred_test_hybrid}
    store[name] = {'pred_test_base': pred_test_base, 'residual_oof': residual_oof}

results_df = pd.DataFrame(results).set_index('Model')
display(results_df)

In [ ]:
# ----------------------------------------------------------------------
# 7.1 (TÙY CHỌN) 1D-CNN. Mặc định TẮT: tích chập trượt trên trục "đặc trưng" của dữ liệu bảng
# không có ý nghĩa vì thứ tự cột là tùy ý. Nếu bật, đầu vào được chuẩn hóa, early stopping dùng
# 15% cảnh muộn nhất trong tập huấn luyện, và GP học phần dư OOF (không phải in-sample).
# ----------------------------------------------------------------------
RUN_CNN = False

if RUN_CNN:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader

    class CNN1D_Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
            self.bn1 = nn.BatchNorm1d(16)
            self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
            self.bn2 = nn.BatchNorm1d(32)
            self.pool = nn.AdaptiveAvgPool1d(8)
            self.dropout = nn.Dropout(0.20)
            self.fc1 = nn.Linear(32 * 8, 64)
            self.fc2 = nn.Linear(64, 1)

        def forward(self, x):
            x = torch.relu(self.bn1(self.conv1(x)))
            x = torch.relu(self.bn2(self.conv2(x)))
            x = self.dropout(self.pool(x).flatten(1))
            return self.fc2(torch.relu(self.fc1(x)))

    def train_cnn(X_tr, y_tr, X_va, y_va, epochs=200, lr=0.001, batch_size=32, patience=25, seed=42):
        torch.manual_seed(seed)
        X_tr_t, y_tr_t = torch.FloatTensor(X_tr).unsqueeze(1), torch.FloatTensor(y_tr).unsqueeze(1)
        X_va_t, y_va_t = torch.FloatTensor(X_va).unsqueeze(1), torch.FloatTensor(y_va).unsqueeze(1)
        loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)
        model = CNN1D_Model()
        crit = nn.MSELoss()
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
        best, best_w, cnt = float('inf'), None, 0
        for ep in range(epochs):
            model.train()
            for bx, by in loader:
                opt.zero_grad()
                loss = crit(model(bx), by)
                if torch.isfinite(loss):
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            model.eval()
            with torch.no_grad():
                vl = crit(model(X_va_t), y_va_t).item()
            if np.isfinite(vl):
                sched.step(vl)
                if vl < best:
                    best, best_w, cnt = vl, copy.deepcopy(model.state_dict()), 0
                else:
                    cnt += 1
            if cnt >= patience:
                break
        if best_w is not None:
            model.load_state_dict(best_w)
        return model

    train_dates = df_train['date'].values

    def cnn_fit_predict(tr_idx, X_new_list):
        sc = StandardScaler().fit(X_train_base[tr_idx])          # chuẩn hóa chỉ bằng dữ liệu huấn luyện
        d = train_dates[tr_idx]
        ud = np.unique(d)
        is_val = d >= ud[int(len(ud) * 0.85)]                      # 15% cảnh muộn nhất làm validation
        Xa, ya = sc.transform(X_train_base[tr_idx]), y_train[tr_idx]
        model = train_cnn(Xa[~is_val], ya[~is_val], Xa[is_val], ya[is_val])
        model.eval()
        with torch.no_grad():
            return [np.clip(model(torch.FloatTensor(sc.transform(X)).unsqueeze(1)).numpy().ravel(), 0, None)
                    for X in X_new_list]

    print('-> Đang huấn luyện 1D-CNN (OOF + full)...')
    cnn_oof = np.zeros(len(y_train))
    for tr, va in splits:
        cnn_oof[va] = cnn_fit_predict(tr, [X_train_base[va]])[0]
    cnn_pred_train, cnn_pred_test = cnn_fit_predict(np.arange(len(y_train)), [X_train_base, X_test_base])

    cnn_resid = y_train - cnn_oof
    cnn_gps = fit_correction(t_tr, xy_tr, cnn_resid)
    cnn_hybrid_te = cnn_pred_test + predict_correction(cnn_gps, t_te, xy_te)

    ev_tr, ev_cv = evaluate_model(y_train, cnn_pred_train), evaluate_model(y_train, cnn_oof)
    results.append(row('1D-CNN', ev_tr, ev_cv, evaluate_model(y_test, cnn_pred_test)))
    results.append(row('1D-CNN + GP', ev_tr, ev_cv, evaluate_model(y_test, cnn_hybrid_te)))
    predictions['1D-CNN'] = {'base': cnn_pred_test, 'hybrid': cnn_hybrid_te}
    store['1D-CNN'] = {'pred_test_base': cnn_pred_test, 'residual_oof': cnn_resid}
else:
    print('RUN_CNN = False -> bỏ qua 1D-CNN.')

**Cell 7.2: Đánh giá cuốn chiếu (rolling-origin)**
- Mô hình nền (cây + Ridge) được fit MỘT LẦN trên train và giữ nguyên. Phần dư của các cảnh test luôn tính từ dự báo nền cố định này.
- Với mỗi cảnh test (theo thứ tự ngày), GP được fit lại bằng phần dư OOF của train cộng với phần dư của các cảnh test ĐÃ QUA, rồi dự báo hiệu chỉnh cho cảnh đó. Cảnh đang dự báo và các cảnh sau nó không bao giờ lọt vào dữ liệu học.
- Cảnh đầu tiên chưa có cảnh nào đứng trước nên bị loại khỏi phép so sánh (`mask`). Mọi cách được so trên cùng tập hàng.
- Giả định vận hành: số đo độ mặn của các cảnh trước đã có sẵn lúc dự báo cảnh sau. Hai baseline: mô hình nền, và nền cộng trung bình phần dư các cảnh đã thấy (không dùng GP). GP phải thắng cả hai mới đáng giữ.
- `Bias = mean(y_thật - y_dự_báo)`: dương nghĩa là đoán thấp.

In [ ]:
test_dates_arr = df_test['date'].values
dates_test = np.unique(test_dates_arr)

def rolling_correction(pred_base, residual_oof):
    corr = np.zeros(len(y_test))
    r_test = y_test - pred_base                      # phần dư đo bằng dự báo nền CỐ ĐỊNH
    for d in dates_test:
        now  = (test_dates_arr == d)
        seen = (test_dates_arr <  d)                 # chỉ các cảnh TRƯỚC d (không dùng <=)
        t  = np.vstack([t_tr,  t_te[seen]])
        xy = np.vstack([xy_tr, xy_te[seen]])
        r  = np.concatenate([residual_oof, r_test[seen]])
        corr[now] = predict_correction(fit_correction(t, xy, r), t_te[now], xy_te[now])
    return corr

def rolling_offset(pred_base):                       # baseline không dùng GP
    out, r_test = pred_base.copy(), y_test - pred_base
    for d in dates_test:
        now  = (test_dates_arr == d)
        seen = (test_dates_arr <  d)
        if seen.any():
            out[now] += r_test[seen].mean()
    return out

mask = test_dates_arr > dates_test[0]                 # bỏ cảnh đầu (chưa có cảnh nào trước nó)
roll_rows = []
for name, s in store.items():
    base = s['pred_test_base']
    corr = rolling_correction(base, s['residual_oof'])
    predictions[name]['rolling'] = np.clip(base + corr, 0, None)
    variants = {'Nền': base,
                'Nền + offset TB': np.clip(rolling_offset(base), 0, None),
                'Nền + GP (λ=1)': np.clip(base + corr, 0, None),
                'Nền + GP (λ=0.5)': np.clip(base + 0.5 * corr, 0, None)}
    for k, p in variants.items():
        ev = evaluate_model(y_test[mask], p[mask])
        roll_rows.append({'Model': name, 'Cách': k, 'R2': ev['R2'], 'Ex_VAR (%)': ev['Ex_VAR (%)'], 'RMSE (‰)': ev['RMSE (‰)'], 'Bias (‰)': ev['Bias (‰)']})

roll_df = pd.DataFrame(roll_rows)
col_order = ['Nền', 'Nền + offset TB', 'Nền + GP (λ=1)', 'Nền + GP (λ=0.5)']
print(f'Đánh giá cuốn chiếu trên {int(mask.sum())} mẫu (từ cảnh thứ 2 trở đi)')
for metric in ['R2', 'Ex_VAR (%)', 'RMSE (‰)', 'Bias (‰)']:
    print('\n' + metric)
    display(roll_df.pivot(index='Model', columns='Cách', values=metric)[col_order])
roll_df.to_csv(f'result_rolling_gp_{RUN_TAG}.csv', index=False)

# Trên TOÀN BỘ test (cảnh đầu chỉ được GP học từ train), để báo cáo song song với bảng tĩnh
all_rows = []
for name in store:
    for k, p in {'Nền': store[name]['pred_test_base'], 'Nền + GP cuốn chiếu': predictions[name]['rolling']}.items():
        ev = evaluate_model(y_test, p)
        all_rows.append({'Model': name, 'Cách': k, 'R2': ev['R2'], 'Ex_VAR (%)': ev['Ex_VAR (%)'],
                         'RMSE (‰)': ev['RMSE (‰)'], 'Bias (‰)': ev['Bias (‰)']})
print('\nTrên TOÀN BỘ 7 cảnh test (cảnh đầu chỉ được GP học từ train):')
display(pd.DataFrame(all_rows).set_index(['Model', 'Cách']))

In [ ]:
# ----------------------------------------------------------------------
# 3. TỔNG HỢP VÀ HIỂN THỊ BẢNG KẾT QUẢ ĐỐI CHUẨN TOÀN DIỆN (đánh giá TĨNH)
# ----------------------------------------------------------------------
results_df = pd.DataFrame(results).set_index('Model')
print('=' * 95)
print('BẢNG SO SÁNH HIỆU NĂNG: đánh giá tĩnh (GP chỉ thấy train). So CV_R2 (OOF) với Test_R2.')
print('=' * 95)
display(results_df)

results_df.to_csv(f'result_hybrid_gp_{RUN_TAG}.csv')
print(f'-> Đã lưu: result_hybrid_gp_{RUN_TAG}.csv và result_rolling_gp_{RUN_TAG}.csv')

**Cell 8: Trực quan hóa kết quả dự báo (Scatter Plot 1:1 & Phân tích sai số Residual)**
- Biểu đồ Scatter Plot phân tán (Observed vs Predicted) so sánh trực tiếp mô hình Đơn lẻ vs Mô hình Lai (+ GP).
- Đánh giá trực quan mức độ cải thiện và khả năng hội tụ dọc theo đường chéo 1:1.

In [ ]:
# Chọn mô hình để trực quan hóa
target_model = 'PSO RF'
if target_model not in predictions:
    target_model = list(predictions.keys())[0]

pred_te_base   = predictions[target_model]['base']
pred_te_hybrid = predictions[target_model]['hybrid']
pred_te_roll   = predictions[target_model]['rolling']

eval_b, eval_h, eval_r = (evaluate_model(y_test, p) for p in (pred_te_base, pred_te_hybrid, pred_te_roll))

panels = [
    (pred_te_base,   f'(a) {target_model} (nền)',               eval_b, '#1f77b4'),
    (pred_te_hybrid, f'(b) {target_model} + GP (đánh giá tĩnh)', eval_h, '#2ca02c'),
    (pred_te_roll,   f'(c) {target_model} + GP (cuốn chiếu)',    eval_r, '#ff7f0e'),
]
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
max_val = max(y_test.max(), *(p.max() for p, *_ in panels)) + 2
for ax, (p, title, ev, col) in zip(axes, panels):
    ax.scatter(y_test, p, alpha=0.6, edgecolors='k', linewidths=0.5, c=col)
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='1:1 Line')
    ax.set_xlabel('Observed Salinity (‰)', fontsize=12)
    ax.set_ylabel('Predicted Salinity (‰)', fontsize=12)
    ax.set_title(f'{title}\nRMSE = {ev["RMSE (‰)"]} ‰, MAE = {ev["MAE (‰)"]} ‰, R² = {ev["R2"]}, Bias = {ev["Bias (‰)"]}', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('scatter_plot_hybrid_gp_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('-> Đã lưu biểu đồ so sánh: scatter_plot_hybrid_gp_comparison.png')

# Bias trung bình theo từng cảnh test: nền vs GP cuốn chiếu
bias_df = pd.DataFrame({
    'date': pd.to_datetime(test_dates_arr).strftime('%Y-%m-%d'),
    'Nền': y_test - pred_te_base,
    'GP cuốn chiếu': y_test - pred_te_roll,
}).groupby('date').mean()
bias_df['Số mẫu'] = pd.Series(pd.to_datetime(test_dates_arr).strftime('%Y-%m-%d')).value_counts().sort_index().values
print('\nBias trung bình theo cảnh (dương = đoán thấp):')
display(bias_df.round(2))

**Cell 9: Ablation – mô hình đang dựa vào những nhóm biến nào?**
Bỏ lần lượt `sin_doy/cos_doy` và `Lat/Long` rồi chạy lại một mô hình (PSO LightGBM, không blend, không GP) để xem R² và bias thay đổi ra sao. Nếu bỏ `sin/cos_doy` mà bias giảm, mô hình đang bám vào mùa của năm 2019.

In [ ]:
print('Cờ USE_SEASONAL hiện tại =', USE_SEASONAL, '(nếu False thì dòng bỏ sin/cos doy không có tác dụng)')

def eval_drop(drop, model=pso_lgbm_model):
    keep = [i for i, f in enumerate(base_cols) if f not in drop]
    m = clone(model).fit(X_train_base[:, keep], y_train)
    ev = evaluate_model(y_test, m.predict(X_test_base[:, keep]))
    return ev['R2'], ev['RMSE (‰)'], ev['Bias (‰)']

for label, drop in {'đầy đủ': [],
                    'bỏ sin/cos doy': ['sin_doy', 'cos_doy'],
                    'bỏ Lat/Long': ['Lat', 'Long'],
                    'bỏ cả sin/cos doy và Lat/Long': ['sin_doy', 'cos_doy', 'Lat', 'Long']}.items():
    r2, rmse, bias = eval_drop(drop)
    print(f'{label:32s} R2={r2:7.3f}  RMSE={rmse:6.3f}  Bias={bias:6.3f}')

**Quy trình chạy thí nghiệm gợi ý**
1. Chạy toàn bộ notebook một lần với cấu hình mặc định (có PSO). Ghi lại bảng tĩnh và bảng cuốn chiếu (đã có thêm Ex_VAR).
2. Cell 6.0 gợi ý `RECENCY_HALF_LIFE` bằng fold tiến. Đặt giá trị đó ở Cell 4 (hoặc chỉ chạy lại dòng `w_train = ...`) rồi chạy lại từ Cell 7.
3. Đặt `USE_SEASONAL = False`, chạy lại Cell 4, rồi chạy từ Cell 6.1 trở đi (không cần chạy lại PSO). Báo cáo cả hai cấu hình, vì fold trong train không đánh giá công bằng được sin/cos_doy.
4. Mỗi lần chạy lưu file `result_*_{RUN_TAG}.csv` riêng, nên các cấu hình không ghi đè lên nhau.
5. Khi báo cáo: nêu rõ bài toán (dự báo sau khi có ảnh của ngày đó, giai đoạn mới), đưa điểm tĩnh và điểm cuốn chiếu, và ghi giả định của cuốn chiếu (số đo của các cảnh trước đã có sẵn lúc dự báo).